In [3]:
import json
from confluent_kafka import Consumer, Producer

BOOTSTRAP = "localhost:9092,localhost:9094,localhost:9096"
LAT_BOUNDS, LON_BOUNDS = (12.80, 13.15), (77.45, 77.80)

producer = Producer({"bootstrap.servers": BOOTSTRAP})
consumer = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "urbanpulse-dlq-validator",
                      "auto.offset.reset": "latest"})
consumer.subscribe(["urbanpulse.bus_gps", "urbanpulse.air_quality"])

def validate(topic, event):
    if topic.endswith("air_quality"):
        if event.get("aqi") is None:
            return "null_value"
        if not (0 <= event["aqi"] <= 500):
            return "out_of_range_aqi"
    if topic.endswith("bus_gps"):
        lat, lon = event.get("lat"), event.get("lon")
        if lat is None or lon is None:
            return "null_value"
        if not (LAT_BOUNDS[0] <= lat <= LAT_BOUNDS[1] and LON_BOUNDS[0] <= lon <= LON_BOUNDS[1]):
            return "invalid_gps"
    return None

while True:
    msg = consumer.poll(0.5)
    if msg is None or msg.error():
        continue
    topic, event = msg.topic(), json.loads(msg.value())
    reason = validate(topic, event)
    if reason:
        payload = {**event, "source_topic": topic, "error_reason": reason}
        producer.produce("urbanpulse.dlq", value=json.dumps(payload).encode())
        producer.poll(0)
        print(f"[DLQ] {topic} -> {reason}")

[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> out_of_range_aqi
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.air_quality -> null_value
[DLQ] urbanpulse.bus_gps -> invalid_gps
[DLQ] urbanpulse.bus_gps -> invalid_gps
[DLQ] urbanpulse.bus_gps -> invalid_gps
[DLQ] urbanpulse.bus_gps -> invalid_gps
[DLQ] urbanpulse.bus_gps -> invalid_gps
[DLQ] urbanpulse.bus_gps -> invalid_gps
[DLQ] urbanpulse.bus_gp

KeyboardInterrupt: 